# Getting started with StableBound

StableBound is a Python package for working with crop statistics across
districts that have been redrawn over time. It solves the practical
problem that every researcher running long-term district-level analyses
hits: **names get reused, districts split and merge, and naive
name-matching against a modern shapefile silently corrupts decades of
data**.

This notebook walks through the core workflow using a tiny synthetic
country bundled with the package, **Exampleland**:

- 5 districts (Alpha, Bravo, Charlie, Delta, Echo) at the 2010 baseline
- A Split in 2014 (Alpha → Alpha North + Alpha South)
- A NameChange in 2016 (Charlie → Charlie Renamed)
- A Merge in 2018 (Delta + Echo → DeltaEcho)
- Stats from 2010 to 2020, with rice area + production per district

Everything fits in a few seconds so you can run, tweak, and re-run.

## What you'll do

1. Load the lineage (events + baseline)
2. Inspect it (snapshots, events, validation)
3. Attach the modern shapefile
4. Build the **Stable Boundary** product -- the paper's main output
5. Aggregate stats onto stable groups
6. Look at the output files
7. Briefly try the **Modern Boundary** product (history rescaled to today's geometry)
8. Where to go next

If you just want the punchline, jump to Section 4.


In [ ]:
# --- 1. Setup ---------------------------------------------------------
#
# If the package is pip-installed this imports directly; running from a
# repo checkout (without installing) falls back to the source tree.
%matplotlib inline
import sys
from pathlib import Path

PKG_ROOT = Path('..').resolve()
try:
    import stablebound
except ImportError:
    sys.path.insert(0, str(PKG_ROOT / 'src'))

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display

# Nicer table display.
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')

# Public surface we'll use today.
from stablebound import Lineage, StableBoundary, ModernBoundary
print('StableBound imported.')

## 2. Load the Exampleland data

The fixture lives in `examples/exampleland/` and ships with three files:

- `relationship_table.csv` -- one row per lineage event (Split, Merge, NameChange...)
- `baseline.csv` -- the units that existed at the start year (2010)
- `modern.geojson` -- the modern district geometries (one polygon each)

Let's look at each before constructing anything.


In [ ]:
# --- 2a. Inspect the raw input files ---------------------------------
EXAMPLELAND = PKG_ROOT / 'examples' / 'exampleland'

print('relationship_table.csv:')
display(pd.read_csv(EXAMPLELAND / 'relationship_table.csv'))
print()

print('baseline.csv:')
display(pd.read_csv(EXAMPLELAND / 'baseline.csv'))
print()

print('modern.geojson (no geometry):')
display(gpd.read_file(EXAMPLELAND / 'modern.geojson').drop(columns='geometry'))


## 3. Construct a Lineage

The `Lineage` class is the package's primary primitive. It bundles the
relationship table + baseline + (optional) name-change log into one
object that knows what units existed in any given year.

For the bundled country (`"IN"`, India) you can write `Lineage("IN")` and
the package finds the canonical files. For a custom country like
Exampleland (`"EX"`), pass explicit paths.

Lineage construction is **lazy**: no disk reads happen here beyond path
existence checks. The relationship-table parsing + validation happens on
first access to `ln.lineage`.


In [ ]:
# --- 3. Build the Lineage ---------------------------------------------
ln = Lineage(
    'EX',
    relationship_table_path=EXAMPLELAND / 'relationship_table.csv',
    baseline_path=EXAMPLELAND / 'baseline.csv',
)
print(ln)   # cheap __repr__ — no validation/parsing triggered yet


## 4. Inspect the lineage

A few things you'll want to peek at:

- The **year range** spanned by the lineage events
- A **snapshot** at any year -- the active units at that point in time
- **Validation issues** -- the package runs 8 data-quality checks
  automatically on first access


In [ ]:
# --- 4a. Year range and validation ------------------------------------
print(f'Year range: {ln.min_year} - {ln.max_year}')
print(f'Years: {list(ln.years)}')
print()
print(ln.validation_report())


In [ ]:
# --- 4b. Snapshots -- what units existed at year T --------------------
#
# Note the event-year convention: an event at year T happens DURING T,
# so snapshot(T) is PRE-event and snapshot(T+1) is post-event.
for yr in (2010, 2014, 2015, 2018, 2019):
    snap = ln.snapshot(year=yr)
    names = ', '.join(snap['unit_name'])
    print(f'{yr}: {len(snap)} units -- {names}')


In [ ]:
# --- 4c. The lineage events themselves ---------------------------------
# `ln.lineage` is the parsed event graph; `ln.lineage.events` is the
# canonical DataFrame.
display(ln.lineage.events)


## 5. Attach the modern shapefile

The Stable Boundary product is anchored to a modern shapefile and the
union-find walk uses it to dissolve into stable groups. For most workflows
you'd run `propose_shapefile_mapping(...)` first to match names, review
the proposal as a CSV, and then `attach_shapefile(..., mapping=...)`.

Exampleland's modern.geojson already has canonical `unit_id`s attached
(no name-matching required), so we can pass it straight through.


In [ ]:
# --- 5. Attach the modern shapefile ----------------------------------
ln.attach_shapefile(EXAMPLELAND / 'modern.geojson')

print(f'Attached: {len(ln.shapefile)} modern polygons')
print(f'  inferred shapefile year: {ln.shapefile_year}')
print(f'  unmatched features:      {len(ln.unmatched_features)}')


## 6. Build the Stable Boundary

This is the paper's main product. For each year in `[target_year, max_year]`:

1. Compute the snapshot of active units (which districts exist this year)
2. Walk the lineage forward from the target_year units, building a
   union-find that groups every unit linked by Split / Merge / Redistribute
3. Dissolve the modern shapefile by stable group → output one `stable_<year>.geojson`

The result is **boundaries that are consistent through history** -- aggregating
stats to them gives a timeseries that's conservation-clean across all
lineage events.


In [ ]:
# --- 6. Build the Stable Boundary ------------------------------------
OUT = PKG_ROOT / 'notebooks' / '_gs_out'
sb = StableBoundary(
    ln,
    target_year=2010,   # anchor the stable boundaries here
    max_year=2020,      # build through this year
    output_dir=OUT,     # all geojsons + audit files land here
)

sb.build_boundaries()
print(f'\n{sb.summary()}')


The `intensive` dict tells the package which derived variables to compute
and from which inputs:

```python
intensive = {"yield_mt_ha": ("rice_production_mt", "rice_area_ha")}
```

`reconcile_mode="off"` is the current default. The reconcile diagnostic
is temporarily disabled (the published window-averaged tests flag clean
post-event reporting handoffs as suspected double-counting at high rates
on real data; the active merge / subtract modes delete legitimate rows
in response). Use `mode="off"` until the diagnostic is reworked.


In [ ]:
# --- 7. Aggregate stats onto stable groups ---------------------------
sb.aggregate_stats(
    stats=EXAMPLELAND / 'stats.csv',
    intensive={'yield_mt_ha': ('rice_production_mt', 'rice_area_ha')},
    reconcile_mode='off',
)

# `get_stats()` returns the aggregated long-form output.
agg = sb.get_stats()
print(f'Aggregated rows: {len(agg)}')
display(agg.head(10))


In [ ]:
# --- Quick view: rice area on a single stable group over time --------
rice = agg[agg['variable'] == 'rice_area_ha'].sort_values(['stable_id', 'year'])

fig, ax = plt.subplots(figsize=(9, 5))
for sid, sub in rice.groupby('stable_id'):
    ax.plot(sub['year'], sub['value'], 'o-', label=sid)
ax.set_xlabel('year')
ax.set_ylabel('rice_area_ha')
ax.set_title('Stable timeseries by stable_id, Exampleland')
ax.legend(loc='best', fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 8. Look at the output files

`StableBoundary.build_boundaries()` writes one `stable_<year>.geojson`
per year plus a bundle of audit files. Let's list them and load one.


In [ ]:
# --- 8a. List output files -------------------------------------------
print(f'Files written to {OUT.relative_to(PKG_ROOT)}:')
for path in sorted(OUT.iterdir()):
    if path.is_file():
        print(f'  {path.name}   ({path.stat().st_size:,} bytes)')


In [ ]:
# --- 8b. Plot the boundaries at the start and end ---------------------
#
# Side-by-side maps showing what the stable groups look like at the start
# year (each unit is its own group) vs the end year (after Split/Merge).
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

for ax, year in zip(axes, (2010, 2020)):
    g = gpd.read_file(OUT / f'stable_{year}.geojson')
    g.plot(ax=ax, column='stable_id', categorical=True, edgecolor='white',
           legend=True, legend_kwds={'fontsize': 8, 'loc': 'best'})
    ax.set_title(f'stable_{year}.geojson', fontsize=11)
    ax.set_axis_off()

plt.tight_layout()
plt.show()


In [ ]:
# --- 8c. Reconciliation diagnostic (currently disabled) ---------------
#
# reconcile_mode="off" is the only accepted value (see Known Issues in the
# README), so the flags frame is empty by construction.
flags = sb.get_reconciliation_flags()
print(f'reconciliation flags: {len(flags)} row(s)')
if len(flags):
    display(flags)
else:
    print('(reconcile diagnostic is disabled — no flags computed.)')

## 9. Briefly: the Modern Boundary

The other product the package builds is the **Modern Boundary** -- the
mirror of Stable. Same input data, opposite framing:

- **Stable** = boundaries pinned to a past year, stats lifted forward
- **Modern** = boundaries pinned to today, stats rescaled backward

Useful when downstream consumers expect data on the modern map (e.g.
joining to a current population layer). The package distributes
historical values to modern units using a 3-tier cascade (seasonal
fractions → total-year fractions → modern area share).


In [ ]:
# --- 9. Modern boundary -----------------------------------------------
mb = ModernBoundary(
    ln,
    target_year=2010,
    output_dir=OUT,    # writes a 'modern/' subdirectory
)
mb.aggregate_stats(
    stats=EXAMPLELAND / 'stats.csv',
    intensive={'yield_mt_ha': ('rice_production_mt', 'rice_area_ha')},
)

modern_stats = mb.get_modern_stats()
print(f'modern rows: {len(modern_stats)}')
display(modern_stats.head(10))

print()
print(f'output files under {OUT.relative_to(PKG_ROOT)}/modern/:')
for path in sorted((OUT / 'modern').iterdir()):
    print(f'  {path.name}')


## 10. Where to go from here

You've now run the full workflow on a tiny synthetic country. The same
API works on real data; the only thing that changes is the input files.

**To switch to a real country:**

```python
# The bundled country: India
ln = Lineage('IN')

# Or any custom country -- pass paths:
ln = Lineage('XX',
             relationship_table_path='path/to/rt.csv',
             baseline_path='path/to/baseline.csv',
             name_change_log_path='path/to/ncl.xlsx')
```

**For real shapefile attachment** (with name-matching):

```python
proposal = ln.propose_shapefile_mapping(
    'modern.geojson', name_column='DISTRICT', coarse_column='STATE',
)
proposal.to_csv('review.csv')    # review in Excel
# ... fix bad matches ...
ln.attach_shapefile('modern.geojson', mapping='review.csv',
                    name_column='DISTRICT')
```

**For the legacy FEWS RT format**:

```python
ln = Lineage.from_legacy_rt('relationshiptable_XX.csv',
                            country='XX', admin_level=1)
# tests/fixtures/rt_convert/relationshiptable_XX.csv is a small real example
```

**Further reading inside the repo:**

- `docs/USAGE.md` -- the practical end-to-end guide
- `docs/methodology.md` -- the paper's algorithms in detail
- `docs/WALKTHROUGH.md` -- the engineering tour of every module
- `examples/README.md` -- worked examples with real output: matching, assessment, the legacy FEWS dialect, the FEWS deliverables